# 04 — Particle Filter (Monte Carlo) Localization

**Section:** Localization · **Mirrors MATLAB:** *Monte Carlo Localization*

The particle filter represents the belief over robot pose with a set of weighted samples (particles).


## Intuition — what's actually going on?

The EKF (notebook 03) assumes your belief about your pose looks like a single Gaussian "blob" — one peak with elliptical uncertainty. That breaks down if you're, say, in a long hallway where you could be anywhere along it — the true belief has multiple peaks.

The **particle filter** drops the Gaussian assumption and represents the belief with hundreds or thousands of *samples* (particles), each one a hypothesis "maybe I'm here". After each motion, you nudge every particle through the motion model with a bit of noise. After each observation, you ask each particle "how likely was that observation if I were you?" and weight it accordingly. Then you **resample** — particles with high weight reproduce, particles with low weight die. Over time the cloud collapses onto the true pose.

It's basically Darwinian evolution for pose hypotheses. The animation in the README shows it beautifully — a cloud of green dots collapses onto the true trajectory as observations come in.


## Analytical derivation

We want to recursively estimate $p(x_t \mid z_{1:t}, u_{1:t})$ where $x_t$ is the pose, $u_t$ is the control input, and $z_t$ is the observation. Bayes-filter recursion:

$$p(x_t \mid z_{1:t}, u_{1:t}) \;\propto\; p(z_t \mid x_t)\;\int p(x_t \mid x_{t-1}, u_t)\,p(x_{t-1} \mid z_{1:t-1}, u_{1:t-1})\,dx_{t-1}$$

The particle filter approximates this with $N$ weighted samples $\{(x_t^{(i)}, w_t^{(i)})\}_{i=1}^N$:

$$p(x_t \mid z_{1:t}, u_{1:t}) \;\approx\; \sum_{i=1}^N w_t^{(i)}\,\delta(x_t - x_t^{(i)})$$

**Sequential Importance Sampling (with motion-model proposal).** At each step:

1. **Predict:** $\tilde x_t^{(i)} \sim p(x_t \mid x_{t-1}^{(i)}, u_t)$  (sample from the motion model)
2. **Weight:** if we use the *motion model* as proposal, the importance weight simplifies to
$$w_t^{(i)} \;\propto\; w_{t-1}^{(i)}\,p(z_t \mid \tilde x_t^{(i)})$$
For Gaussian observation noise $z = h(x) + v$, $v \sim \mathcal{N}(0, \Sigma)$:
$$p(z_t \mid \tilde x_t^{(i)}) \;\propto\; \exp\!\Bigl(-\tfrac{1}{2}\,(z_t - h(\tilde x_t^{(i)}))^T \Sigma^{-1} (z_t - h(\tilde x_t^{(i)}))\Bigr)$$
Compute in log-space then exponentiate after subtracting the maximum (avoids underflow).
3. **Resample:** if effective sample size $N_\text{eff} = 1 / \sum (w^{(i)})^2$ falls below threshold (or every step). Systematic resampling preserves diversity with $O(N)$ cost:
$$U^{(i)} = \frac{i - 1 + U_0}{N},\quad U_0 \sim \text{Unif}(0,1)$$
and pick particle $j$ such that $\sum_{k=1}^{j-1} w^{(k)} < U^{(i)} \le \sum_{k=1}^{j} w^{(k)}$.

After resampling all weights are reset to $1/N$.

### Compatibility check — math ↔ code

| Step | Code |
|---|---|
| Predict $\tilde x_t^{(i)}$ via motion model + noise | `particles += u + np.random.randn(N, 3) * np.array([0.08, 0.08, 0.04])` |
| $h(x) = \|\ell_j - x\|$ (range to landmark $j$) | `expected = np.linalg.norm(landmarks[:, None, :] - particles[None, :, :2], axis=2)` |
| $(z - h(x))/\sigma$ standardized residual | `err = (z[:, None] - expected) / range_noise` |
| $\log w \propto -\tfrac{1}{2}\sum_j r_j^2$ | `log_w = -0.5 * np.sum(err ** 2, axis=0)` |
| Normalize after max-subtract | `w = np.exp(log_w - log_w.max()); w /= w.sum()` |
| Systematic resampling | `idx = np.searchsorted(np.cumsum(w), (np.arange(N) + np.random.random()) / N)` |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

T = 120
dt = 0.1

# Figure-8 ground-truth path
true_x = np.zeros((T, 3))
for t in range(T):
    a = t * dt
    true_x[t] = [5 * np.sin(a), 3 * np.sin(2 * a), a]

landmarks = np.array([[6, 0], [-6, 0], [0, 4], [0, -4], [4, 4], [-4, -4]])


In [ ]:
N = 600
# Initial particles spread around the origin
particles = np.zeros((N, 3))
particles[:, :2] = np.random.uniform(-2, 2, (N, 2))
particles[:, 2] = np.random.uniform(-np.pi, np.pi, N)

est_hist, particle_hist = [], []
range_noise = 0.25

for t in range(T):
    u = (true_x[t] - true_x[t - 1]) if t > 0 else np.zeros(3)
    particles += u + np.random.randn(N, 3) * np.array([0.08, 0.08, 0.04])

    # Range observations to each landmark
    z = np.linalg.norm(landmarks - true_x[t, :2], axis=1) + np.random.randn(len(landmarks)) * range_noise
    expected = np.linalg.norm(landmarks[:, None, :] - particles[None, :, :2], axis=2)  # (M, N)
    err = (z[:, None] - expected) / range_noise
    log_w = -0.5 * np.sum(err ** 2, axis=0)
    w = np.exp(log_w - log_w.max())
    w /= w.sum()

    # COUNCIL FIX (pass 4, Wald): resampling every step is a deliberate
    # simplification — because we recompute weights from scratch each step
    # (no carry-over from w_{t-1}), the SIR recursion collapses to
    # w_t ∝ p(z_t | x_t) and the post-resample weights become uniform.
    # Production SIR carries weights and conditionally resamples when
    # N_eff = 1/Σw² drops below N/2; we print N_eff to show it.
    N_eff = 1.0 / np.sum(w ** 2)
    idx = np.searchsorted(np.cumsum(w), (np.arange(N) + np.random.random()) / N)
    particles = particles[idx]

    est_hist.append(particles.mean(axis=0))
    if t in (0, T // 3, 2 * T // 3, T - 1):
        particle_hist.append((t, particles.copy()))

est_hist = np.array(est_hist)
err = np.linalg.norm(est_hist[:, :2] - true_x[:, :2], axis=1)
print(f"Mean position error: {err.mean():.3f} m")


In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 10))
for ax, (t, p) in zip(axs.flat, particle_hist):
    ax.scatter(p[:, 0], p[:, 1], c='g', s=4, alpha=0.4, label='Particles')
    ax.plot(true_x[:t + 1, 0], true_x[:t + 1, 1], 'b-', lw=2, label='True')
    ax.plot(est_hist[:t + 1, 0], est_hist[:t + 1, 1], 'r--', lw=1, label='Estimate')
    ax.scatter(landmarks[:, 0], landmarks[:, 1], c='k', marker='*', s=140)
    ax.set_title(f't = {t * dt:.1f}s')
    ax.set_aspect('equal'); ax.grid(); ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()


## References & rigor notes

**Theorem** (Almost-sure convergence; Crisan & Doucet, 2002).
*Under regularity conditions on the proposal and likelihood, the empirical measure of $N$ particles converges weakly to the true posterior as $N \to \infty$, with $L^p$ error $O(1/\sqrt{N})$.*

**Effective sample size.** Without resampling, weight concentrates on a single particle ("degeneracy"). Standard practice: monitor $N_\text{eff} = 1 / \sum_i (w_i)^2$ and resample when $N_\text{eff} < N/2$. Our notebook resamples *every step* — a simplification valid here because we recompute weights from scratch each step (no $w_{t-1}$ carry-over), collapsing the SIR recursion to $w_t \propto p(z_t \mid x_t)$.

**Posterior Cramér-Rao bound (PCRB).** As with the EKF (notebook 03), the PCRB (Tichavský-Muravchik-Nehorai 1998) is the achievability benchmark; the particle filter approaches it as $N \to \infty$ in the linear-Gaussian limit and bounds the noise-floor performance in nonlinear cases.

**Observation note.** This notebook intentionally uses **range only** (no bearing) to expose heading-unobservability — heading drifts via motion-model noise alone. With range + bearing (as in notebook 03), heading is observable. Mean estimator collapses multimodal beliefs; for multimodal targets prefer MAP `particles[np.argmax(w)]` or KDE-mode.

**Sample impoverishment.** Resampling *too often* concentrates particles onto the highest-weight one and loses diversity. Mitigations: regularized PF (add noise after resampling), MCMC moves, or Rao-Blackwellization (marginalize linear-Gaussian sub-states).

**Complexity.** Per step: $O(N M)$ for $N$ particles and $M$ observations; systematic resampling is $O(N)$.

**References.**
- Doucet, A., Godsill, S., & Andrieu, C. (2000). *On sequential Monte Carlo sampling methods for Bayesian filtering*. Statistics and Computing, 10(3), 197-208.
- Crisan, D., & Doucet, A. (2002). *A survey of convergence results on particle filtering methods for practitioners*. IEEE Trans. Signal Processing, 50(3).
- Thrun, S., Burgard, W., & Fox, D. (2005). *Probabilistic Robotics*, MIT Press, ch. 4.
